In [1]:
!pip install spacy

Defaulting to user installation because normal site-packages is not writeable
  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
   ---------------------------------------- 0.0/14.4 MB ? eta -:--:--
   - -------------------------------------- 0.5/14.4 MB 4.4 MB/s eta 0:00:04
   -------- ------------------------------- 2.9/14.4 MB 8.2 MB/s eta 0:00:02
   ------------ --------------------------- 4.5/14.4 MB 7.3 MB/s eta 0:00:02
   ---------------- ----------------------- 5.8/14.4 MB 6.6 MB/s eta 0:00:02
   ------------------ --------------------- 6.8/14.4 MB 6.3 MB/s eta 0:00:02
   --------------------- ------------------ 7.6/14.4 MB 6.0 MB/s eta 0:00:02
   ------------------------ --------------- 8.7/14.4 MB 5.6 MB/s eta 0:00:02
   ------------------------- -------------


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MaxAbsScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import spacy
import optuna
import mlflow
import mlflow.sklearn

# MLflow / DagShub tracking setup
os.environ["MLFLOW_TRACKING_USERNAME"] = "Roy7721"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "bc912b7d58bd2aca05abdc192e010414493a3886"
mlflow.set_tracking_uri("https://dagshub.com/Roy7721/yt_comment_analysis.mlflow")
mlflow.set_experiment("Custom Features - LogisticRegression HP Tuning")

C:\Users\Dell\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/07/21 06:52:52 INFO mlflow.tracking.fluent: Experiment with name 'Custom Features - LogisticRegression HP Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/39879ebee5a848d1950a26b5f5756b98', creation_time=1784595173277, effective_trace_archival_retention=None, experiment_id='13', last_update_time=1784595173277, lifecycle_stage='active', name='Custom Features - LogisticRegression HP Tuning', tags={}, trace_location=None, workspace='default'>

In [4]:
# Load dataset
dataset = pd.read_csv('dataset.csv')

# Drop rows with NaN values in 'clean_comment'
cleaned_dataset = dataset.dropna()

In [5]:
# Separate features and target
X_cleaned = cleaned_dataset['clean_comment']
y_cleaned = cleaned_dataset['category']

# Split the cleaned data into train and test sets (80-20 split)
X_train_cleaned, X_test_cleaned, y_train_cleaned, y_test_cleaned = train_test_split(X_cleaned, y_cleaned, test_size=0.2, random_state=42)


In [11]:
# Load spacy language model for POS tagging
nlp = spacy.load('en_core_web_sm')

In [ ]:
# All 17 universal POS tags, in a FIXED order.
# Looping over this (instead of set(pos_tags)) guarantees every comment produces the
# same 17 POS columns in the same order -> train, test, and future data always line up.
UNIVERSAL_POS = ['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM',
                 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']

# Function to extract custom features
def extract_custom_features(text):
    doc = nlp(text)
    word_list = [token.text for token in doc]

    # 1. Comment Length (number of characters)
    comment_length = len(text)

    # 2. Word Count
    word_count = len(word_list)

    # 3. Average Word Length
    avg_word_length = sum(len(word) for word in word_list) / word_count if word_count > 0 else 0

    # 4. Unique Word Count
    unique_word_count = len(set(word_list))

    # 5. Lexical Diversity
    lexical_diversity = unique_word_count / word_count if word_count > 0 else 0

    # 6. Count of POS Tags
    pos_count = len([token.pos_ for token in doc])

    # 7. Proportion of POS Tags - loop over the FIXED list (not set()) so every comment
    #    yields all 17 columns in the same order; tags not present come out as 0.
    pos_tags = [token.pos_ for token in doc]
    if word_count > 0:
        pos_proportion = {tag: pos_tags.count(tag) / word_count for tag in UNIVERSAL_POS}
    else:
        pos_proportion = {tag: 0 for tag in UNIVERSAL_POS}

    return {
        'comment_length': comment_length,
        'word_count': word_count,
        'avg_word_length': avg_word_length,
        'unique_word_count': unique_word_count,
        'lexical_diversity': lexical_diversity,
        'pos_count': pos_count,
        **pos_proportion  # Flattening the POS proportions
    }

In [ ]:
# Apply the custom feature extraction
train_custom_features = pd.DataFrame([extract_custom_features(text) for text in X_train_cleaned])
test_custom_features = pd.DataFrame([extract_custom_features(text) for text in X_test_cleaned])

In [14]:
train_custom_features.head()

,comment_length,word_count,avg_word_length,unique_word_count,lexical_diversity,pos_count,AUX,VERB,NOUN,ADV,...,ADJ,PART,NUM,INTJ,CCONJ,SCONJ,X,DET,PUNCT,SYM
0,51,7,6.428571,7,1.000000,7,0.142857,0.428571,0.142857,0.142857,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,36,6,5.166667,6,1.000000,6,NaN,0.333333,0.333333,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,64,9,6.222222,9,1.000000,9,NaN,0.222222,0.444444,0.111111,...,0.222222,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,108,15,6.266667,14,0.933333,15,NaN,0.200000,0.466667,NaN,...,0.200000,0.066667,0.066667,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6,1,6.000000,1,1.000000,1,NaN,NaN,1.000000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
# Replace NaN values in POS tag proportions with 0
train_custom_features.fillna(0, inplace=True)
test_custom_features.fillna(0, inplace=True)

In [16]:
test_custom_features.isnull().sum()

comment_length       0
word_count           0
avg_word_length      0
unique_word_count    0
lexical_diversity    0
pos_count            0
ADJ                  0
VERB                 0
NOUN                 0
PRON                 0
ADP                  0
AUX                  0
PROPN                0
ADV                  0
NUM                  0
CCONJ                0
INTJ                 0
DET                  0
PART                 0
SCONJ                0
X                    0
PUNCT                0
SYM                  0
dtype: int64

In [17]:
# Apply TfidfVectorizer with bigram setting and max_features=10000
ngram_range = (1, 2)
max_features = 10000
tfidf = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X_train_tfidf = tfidf.fit_transform(X_train_cleaned)
X_test_tfidf = tfidf.transform(X_test_cleaned)

In [18]:
# Convert TF-IDF to DataFrame
X_train_tfidf_df = pd.DataFrame(X_train_tfidf.toarray(), columns=tfidf.get_feature_names_out())
X_test_tfidf_df = pd.DataFrame(X_test_tfidf.toarray(), columns=tfidf.get_feature_names_out())

In [ ]:
# Combine TF-IDF and custom features
X_train_combined = pd.concat([X_train_tfidf_df.reset_index(drop=True), train_custom_features.reset_index(drop=True)], axis=1)
X_test_combined = pd.concat([X_test_tfidf_df.reset_index(drop=True), test_custom_features.reset_index(drop=True)], axis=1)

# Align test columns to train columns (order AND set).
# The spaCy POS-proportion columns come from set(pos_tags), so train/test can end up in
# different column orders -> sklearn raises "feature names must be in the same order" at predict.
# reindex fixes the order, adds any POS column missing from test (as 0), and drops extras.
X_test_combined = X_test_combined.reindex(columns=X_train_combined.columns, fill_value=0)

In [20]:
X_train_combined

,000,000 000,000 crore,000 people,000 rupee,100,100 crore,100 time,100 year,1000,...,ADJ,PART,NUM,INTJ,CCONJ,SCONJ,X,DET,PUNCT,SYM
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.222222,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.200000,0.066667,0.066667,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29324,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.250000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
29325,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.100000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
29326,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
29327,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.189189,0.000000,0.000000,0.054054,0.0,0.0,0.0,0.0,0.0,0.0


In [21]:
X_train = X_train_combined
X_test = X_test_combined
y_train = y_train_cleaned
y_test = y_test_cleaned

In [22]:
# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test, params, trial_number, log_model=False):
    with mlflow.start_run():
        # Log model type and trial number
        mlflow.set_tag("mlflow.runName", f"Trial_{trial_number}_{model_name}_")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)
        mlflow.log_param("vectorizer_type", "TfidfVectorizer")
        mlflow.log_param("ngram_range", str(ngram_range))
        mlflow.log_param("vectorizer_max_features", max_features)
        mlflow.log_param("scaler", "MaxAbsScaler")

        # Log hyperparameters
        for key, value in params.items():
            mlflow.log_param(key, value)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Only log the model artifact for the best run to stay within DagShub free tier limits
        if log_model:
            # Persist locally first so a long train is never lost to a logging failure
            joblib.dump(model, f"{model_name}_best_model.joblib")
            try:
                mlflow.sklearn.log_model(
                    model, f"{model_name}_model",
                    serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
                )
            except Exception as e:
                print(f"[warn] model artifact logging failed (metrics/params + joblib file are safe): {e}")

        return accuracy

In [23]:
# Step 6: Optuna objective function for Logistic Regression
def objective_logreg(trial):
    # Hyperparameter space to explore
    C = trial.suggest_float('C', 1e-4, 10.0, log=True)  # inverse regularization strength
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])
    # Class imbalance handling - let Optuna decide whether balancing helps
    class_weight = trial.suggest_categorical('class_weight', [None, 'balanced'])

    # Log trial parameters
    params = {
        'C': C,
        'penalty': penalty,
        'class_weight': class_weight,
        'solver': 'saga',   # saga supports both l1 and l2
        'max_iter': 1000
    }

    # MaxAbsScaler -> LogisticRegression pipeline (scaler fit on train folds only, no leakage)
    model = Pipeline([
        ('scaler', MaxAbsScaler()),
        ('clf', LogisticRegression(C=C, penalty=penalty, class_weight=class_weight,
                                   solver='saga', max_iter=1000, random_state=42))
    ])

    # Log each trial as a separate run in MLflow
    accuracy = log_mlflow("LogisticRegression", model, X_train, X_test, y_train, y_test, params, trial.number)

    return accuracy




In [24]:
# Step 7: Run Optuna for Logistic Regression, log the best model, and plot the importance of each parameter
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_logreg, n_trials=50)  # 50 trials

    # Get the best parameters
    best_params = study.best_params
    best_model = Pipeline([
        ('scaler', MaxAbsScaler()),
        ('clf', LogisticRegression(C=best_params['C'],
                                   penalty=best_params['penalty'],
                                   class_weight=best_params['class_weight'],
                                   solver='saga', max_iter=1000, random_state=42))
    ])

    # Log the best model with MLflow (only this run uploads the model artifact)
    log_mlflow("LogisticRegression", best_model, X_train, X_test, y_train, y_test, best_params, "Best", log_model=True)

    # Plot parameter importance
    optuna.visualization.plot_param_importances(study).show()

    # Plot optimization history
    optuna.visualization.plot_optimization_history(study).show()

    # Return the best model so it can be inspected outside this function
    return best_model

In [ ]:
# Run the experiment for Logistic Regression
best_model = run_optuna_experiment()

In [ ]:
best_model